In [1]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, confusion_matrix

hidden_test_df = pd.read_csv('hidden_test_with_labels.csv')

model_path = "./model_checkpoint"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

hidden_test_ds = Dataset.from_pandas(hidden_test_df)

def tokenize_func(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

hidden_test_encoded = hidden_test_ds.map(tokenize_func, batched=True)

trainer = Trainer(model=model)
predictions = trainer.predict(hidden_test_encoded)
predicted_labels = np.argmax(predictions.predictions, axis=-1)

acc = accuracy_score(hidden_test_df['label'], predicted_labels)
cm = confusion_matrix(hidden_test_df['label'], predicted_labels)

print(f"Total Accuracy on Hidden Test Data: {acc * 100:.2f}%")
print(f"Confusion Matrix:\n{cm}")

hidden_test_df['predicted_label'] = predicted_labels
submission_df = hidden_test_df[['id', 'predicted_label']]
submission_df.to_csv('hidden_test_predictions.csv', index=False)

print("\nSuccess! Saved final predictions to hidden_test_predictions.csv")

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 600/600 [00:01<00:00, 492.37 examples/s]
/usr/local/python/3.12.1/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Total Accuracy on Hidden Test Data: 53.33%
Confusion Matrix:
[[ 71 229]
 [ 51 249]]

Success! Saved final predictions to hidden_test_predictions.csv
